# nano-prover SFT on Colab (free T4 GPU)

The Colab kernel runs on a **remote Google GPU machine** (files live under `/content`, not your laptop).
The `.py` scripts that actually run are the ones **cloned by cell 2** — so whenever the code was updated,
**re-run cell 2** (it force-pulls the latest) before the others. **Prereq:** commit + push first.

In [ ]:
# 1. confirm we have a GPU (must show Tesla T4 / similar; 'command not found' => CPU runtime, redo as GPU)
!nvidia-smi

In [ ]:
# 2. clone + FORCE to the latest main (re-run whenever the code changed).  No nesting.
#    NOTE: this keeps untracked files like training/out (your trained adapter) — it does NOT delete them.
#    Private repo: replace the URL with https://<GITHUB_TOKEN>@github.com/FELIXGAO0315/Conjecture-Gym.git
import os
os.chdir('/content')
if not os.path.isdir('/content/Conjecture-Gym'):
    !git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git
os.chdir('/content/Conjecture-Gym')
!git fetch -q --depth 1 origin main && git reset -q --hard origin/main
assert os.path.exists('output/train_corpus.jsonl'), 'corpus not found — did you push it?'
print('cwd:', os.getcwd(), '| HEAD:', end=' ')
!git log --oneline -1

In [ ]:
# 3. install the SFT stack.  (torchao is handled inside sft_train.py — it removes Colab's old one.)
!pip install -q -U transformers trl peft datasets accelerate

In [ ]:
# 4. build train/test splits from the corpus
!python training/prepare_data.py

In [ ]:
# 5. LoRA SFT — defaults: 1 epoch, max_seq 1024, grad-ckpt off  => ~335 steps, ~30 min on T4.
#    Faster first pass (~10 min): !SFT_MODEL=Qwen/Qwen2.5-Coder-0.5B-Instruct python training/sft_train.py
#    OOM? !SFT_GRAD_CKPT=1 python training/sft_train.py   |   more epochs? SFT_EPOCHS=3
!python training/sft_train.py

## Evaluate — generate proofs HERE (GPU), verify LOCALLY (Lean)
The trained adapter is 264 MB — don't move it.  Instead generate proofs on Colab and push back the
tiny `preds.jsonl`; the Lean-kernel scoring runs on your laptop (that's where the Mathlib workspace is).

In [ ]:
# 6. generate proofs for 100 held-out statements with your fine-tuned model -> training/preds.jsonl
!python training/gen_preds.py --n 100 --k 1

In [ ]:
# 7. push the SMALL preds.jsonl back to GitHub (needs a token with repo write).  Then pull it locally.
TOKEN = 'ghp_xxx'   # GitHub Personal Access Token (Settings -> Developer settings -> Tokens, scope: repo)
!git add -f training/preds.jsonl
!git -c user.email=you@example.com -c user.name=you commit -m 'eval preds'
!git push https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git main

Then on your **local** machine (where the Lean verifier lives):
```bash
git pull
python training/verify_preds.py     # real Lean-verified pass@k
```